# Logit Dynamics

How the output prediction evolves through computation. Goes beyond logit lens (which just projects) to analyze the dynamics of prediction formation — flips, stability, commitment timing, and per-component contributions.

**References:**
- Nostalgebraist (2020) "Interpreting GPT: the Logit Lens"
- Din et al. (2023) "Jump to Conclusions: Short-Cutting Transformers"
- Geva et al. (2022) "Transformer Feed-Forward Layers Build Predictions"

## Why This Matters

Logit dynamics tracks how the model's prediction changes layer by layer — when do logit flips occur, when does the prediction stabilize, and which components cause the decisive changes? This extends the logit lens from a static snapshot to a dynamic view of prediction formation.

**Key references:**
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.logit_dynamics import (
    logit_flip_analysis,
    prediction_stability_across_layers,
    commitment_timing,
    logit_contribution_by_component,
    top_k_trajectory,
)

# Create a small model
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35])
print(f"Model: {cfg.n_layers} layers, d_model={cfg.d_model}")

## 1. Logit Flip Analysis

Where does the top-1 prediction change between layers? Each flip represents a computational phase transition where the model changes its "mind" about what token to predict.

In [ ]:
flips = logit_flip_analysis(model, tokens, pos=-1)

print(f"Layer predictions: {flips['layer_predictions']}")
print(f"Flip layers: {flips['flip_layers']}")
print(f"Total flips: {flips['n_flips']}")
print(f"Final prediction: token {flips['final_prediction']}")
print(f"First correct layer: {flips['first_correct_layer']}")

## 2. Prediction Stability

How stable is the top-k prediction set across layers? High stability means the model converges early; low stability means heavy late-stage computation.

In [ ]:
stability = prediction_stability_across_layers(model, tokens, top_k=5)

for l in range(cfg.n_layers):
    print(f"Layer {l}: overlap_with_final={stability['overlap_with_final'][l]:.4f}")

print(f"\nConsecutive stability scores: {stability['stability_scores']}")
print(f"Mean stability: {stability['mean_stability']:.4f}")
print(f"Final top-5: {stability['final_top_k']}")

## 3. Commitment Timing

When does the model "commit" to its final prediction? Tracks the probability of the final output token at each layer to find the commitment point.

In [ ]:
commit = commitment_timing(model, tokens, confidence_threshold=0.3)

for l in range(cfg.n_layers):
    print(f"Layer {l}: confidence={commit['layer_confidence'][l]:.4f}")

print(f"\nCommitment layer (>0.3 prob): {commit['commitment_layer']}")
print(f"Confidence growth rate: {commit['confidence_growth_rate']:.4f}/layer")
print(f"Max confidence jump at layer: {commit['max_confidence_jump']}")
print(f"Final confidence: {commit['final_confidence']:.4f}")

## 4. Logit Contribution by Component

Decompose a target token's logit into contributions from each attention layer and MLP layer. Identifies whether attention or MLP is primarily responsible.

In [ ]:
target = 42
contrib = logit_contribution_by_component(model, tokens, target_token=target)

print(f"Target token: {target}")
print(f"Total logit: {contrib['total_logit']:.4f}")
print(f"Embedding contribution: {contrib['embedding_contribution']:.4f}")
print(f"\nPer-layer contributions:")
for l in range(cfg.n_layers):
    print(f"  Layer {l}: attn={contrib['attn_contributions'][l]:.4f}, "
          f"mlp={contrib['mlp_contributions'][l]:.4f}")

print(f"\nDominant component: {contrib['dominant_component']}")

## 5. Top-k Trajectory

Track how the probability of the final top-k tokens evolves through layers. Visualizes the "race" between candidate predictions.

In [ ]:
traj = top_k_trajectory(model, tokens, top_k=5)

print(f"Tracked tokens: {traj['tracked_tokens']}")
print(f"Convergence layer (top-1 first leads): {traj['convergence_layer']}")
print(f"Final probabilities: {[f'{p:.4f}' for p in traj['final_probabilities']]}")
print(f"\nProbability trajectories:")
for t in traj['tracked_tokens'][:3]:  # Show top 3
    probs = [f"{p:.4f}" for p in traj['probability_trajectories'][t]]
    print(f"  Token {t}: {probs}")